In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [1]:
import torch
import os
import numpy as np

from pathlib import Path

print(torch.cuda.is_available())

True


In [2]:
from pathlib import Path

PROJECT_ROOT_STR = "/content/drive/MyDrive/Indic-Multimodal-NMT"
PROJECT_ROOT = Path(
    PROJECT_ROOT_STR
)

IMAGE_DIR = PROJECT_ROOT / "data/raw/flickr30k/images"

print(IMAGE_DIR.exists())
print(len(list(IMAGE_DIR.glob("*.jpg"))))

True
31014


In [5]:
!pip install -q detectron2@git+https://github.com/facebookresearch/detectron2.git

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [14]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor


cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
    )
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
)

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.DEVICE = "cuda"


predictor = DefaultPredictor(cfg)

In [15]:
model = predictor.model
model.eval()

GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): Sequential(
        (0): BottleneckBlock

In [26]:
import cv2
def extract_features(image_path, max_regions=36):

    image = cv2.imread(
        str(image_path)
    )

    height, width = image.shape[:2]


    inputs = [
        {
            "image": torch.as_tensor(
                image.astype("float32").transpose(2,0,1)
            ),
            "height": height,
            "width": width
        }
    ]

    with torch.no_grad():

        images = model.preprocess_image(inputs)

        features = model.backbone(images.tensor)

        proposals, _ = model.proposal_generator(
            images,
            features,
            None
        )

        proposal_boxes = [
            p.proposal_boxes
            for p in proposals
        ]

        # ROI pooled features
        pooled_features = model.roi_heads.box_pooler(
            [features[f] for f in model.roi_heads.in_features],
            proposal_boxes
        )

        # Pass through ROI box head
        roi_features = model.roi_heads.box_head(
            pooled_features
        )


    roi_features = roi_features.cpu().numpy()

    if roi_features.shape[0] > max_regions:
        roi_features = roi_features[:max_regions]

    return roi_features

In [23]:
from pathlib import Path
import torch.nn.functional as F

image_path = list(
    IMAGE_DIR.glob("*.jpg")
)[0]


features = extract_features(
    image_path
)


print(features.shape)

# 36  → number of detected regions/proposals
# 256 → feature channels
# 7x7 → spatial feature map

# convert to (36,1024) as
# mBART fusion layers can project it easily
# memory is lower
# enough visual information for translation




(36, 256, 7, 7)


In [19]:
FEATURE_DIR =  PROJECT_ROOT / "data/features/flickr30k"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [20]:
feature_file = (
    FEATURE_DIR /
    (image_path.stem + ".npy")
)

np.save(
    feature_file,
    features
)

print(feature_file)

/content/drive/MyDrive/Indic-Multimodal-NMT/data/features/flickr30k/7828338600.npy


In [24]:
print(model.roi_heads.box_head)

FastRCNNConvFCHead(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12544, out_features=1024, bias=True)
  (fc_relu1): ReLU()
  (fc2): Linear(in_features=1024, out_features=1024, bias=True)
  (fc_relu2): ReLU()
)


In [27]:
features = extract_features(image_path)

print(features.shape)

(36, 1024)


In [28]:
from pathlib import Path

FEATURE_DIR = PROJECT_ROOT / "data/features/detectron2"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(FEATURE_DIR)

/content/drive/MyDrive/Indic-Multimodal-NMT/data/features/detectron2


In [29]:
import numpy as np

feature_path = FEATURE_DIR / f"{image_path.stem}.npy"

np.save(
    feature_path,
    features.astype(np.float32)
)

print(feature_path)

/content/drive/MyDrive/Indic-Multimodal-NMT/data/features/detectron2/7828338600.npy


In [30]:
loaded = np.load(feature_path)

print(loaded.shape)

(36, 1024)


In [31]:
from tqdm import tqdm

image_files = sorted(
    IMAGE_DIR.glob("*.jpg")
)

print(len(image_files))

31014


In [ ]:
for image_path in tqdm(image_files):

    feature_path = (
        FEATURE_DIR /
        f"{image_path.stem}.npy"
    )

    if feature_path.exists():
        continue

    try:

        features = extract_features(image_path)

        np.save(
            feature_path,
            features.astype(np.float32)
        )

    except Exception as e:

        print(
            f"Error with {image_path.name}: {e}"
        )

 25%|██▍       | 7672/31014 [56:12<2:40:49,  2.42it/s]